# BraTS 2021 3D Brain Tumor Segmentation Baseline

This notebook implements an end-to-end 3D multi-modal MRI brain tumor segmentation baseline on **BraTS 2021** using **MONAI** and **PyTorch**.

### Adaptive Design
- **Dual-Environment Support**: Seamlessly executes on both **Local Machine** and **Google Colab** via auto-detection.
- **Zero Hardcoded Paths**: Dataset directories and project paths are resolved dynamically without hardcoding OS-specific paths.
- **Modular Architecture**: All core logic is separated into reusable modules under `src/`.
- **Standard Targets**: Predicts **Whole Tumor (WT)**, **Tumor Core (TC)**, and **Enhancing Tumor (ET)** using 3-channel sigmoid outputs.

## 1. Environment Detection and Setup

This cell automatically detects whether the runtime is **Google Colab** or a **Local Environment**.
In Colab, it will install required packages and mount Google Drive if needed.

In [ ]:
import os
import sys
from pathlib import Path

# 1. Auto-detect environment
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
print(f"Environment detected: {'Google Colab' if IS_COLAB else 'Local Environment'}")

# 2. Branching setup for Google Colab
if IS_COLAB:
    print("Running Colab initialization...")
    # Check GPU info
    !nvidia-smi

    # Install dependencies
    !pip install -q monai nibabel matplotlib scikit-learn tqdm

    # Mount Google Drive if accessible
    try:
        from google.colab import drive
        if not os.path.exists("/content/drive"):
            drive.mount("/content/drive")
            print("Google Drive mounted at /content/drive")
    except Exception as e:
        print(f"Drive mount skipped or already mounted: {e}")
else:
    print("Running in local environment; skipping Colab package installation and Drive mount.")

## 2. Dynamic Project Root and Path Resolution

Locate the project root dynamically, add it to `sys.path`, and import the modules from `src`.

In [ ]:
# Locate project root containing 'src' dynamically
def find_project_root() -> Path:
    # Check current directory and parents
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "src" / "config.py").is_file():
            return base.resolve()
    # If running in Colab, search /content
    if IS_COLAB and Path("/content").exists():
        for candidate in Path("/content").rglob("src/config.py"):
            return candidate.parent.parent.resolve()
    return Path.cwd().resolve()

PROJECT_ROOT = find_project_root()
print(f"Resolved Project Root: {PROJECT_ROOT}")

# Change working directory to project root
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import modular components
import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import BaselineConfig, get_default_config, resolve_dataset_dir
from src.eda import (
    scan_brats_dataset,
    compute_volume_statistics,
    analyze_label_distribution,
    plot_patient_modalities
)
from src.dataset import build_dataloaders, get_brats_data_dicts
from src.model import build_unet_model, get_model_summary
from src.loss_metric import build_loss_fn, BraTSMetric
from src.trainer import BraTSTrainer
from src.utils import set_seed, plot_training_curves, visualize_predictions, load_checkpoint

print("All modular components from 'src' successfully imported.")

## 3. Dynamic Dataset Discovery and Configuration

The dataset path is resolved dynamically without any hardcoded paths.
You can specify `CUSTOM_DATASET_PATH` if your data is stored in a non-standard location, or leave it as `None` to auto-discover.

In [ ]:
# Optional custom path override: set to None for automatic discovery
# Example Colab custom path: '/content/drive/MyDrive/MyBraTSData/BraTS2021_Training_Data'
CUSTOM_DATASET_PATH = None

# Dynamically resolve dataset path
DATASET_DIR = resolve_dataset_dir(CUSTOM_DATASET_PATH)
print(f"Resolved Dataset Directory: {DATASET_DIR}")

# Initialize configuration with resolved paths
config = get_default_config(data_dir=DATASET_DIR)
set_seed(config.seed)

# Ensure checkpoint and output directories exist inside project root
os.makedirs(config.checkpoint_dir, exist_ok=True)
os.makedirs(config.output_dir, exist_ok=True)

print(f"Execution Device: {config.device}")
print(f"Checkpoint Directory: {os.path.abspath(config.checkpoint_dir)}")
print(f"Output Directory: {os.path.abspath(config.output_dir)}")
print(f"Patch ROI Size: {config.roi_size}")
print(f"Input Modalities: {config.in_channels} channels (FLAIR, T1, T1ce, T2)")
print(f"Output Classes: {config.out_channels} channels (TC, WT, ET)")

## 4. Exploratory Data Analysis (EDA)

Scan the discovered dataset directory, verify patient cases, and inspect volume metadata and label distributions.

In [ ]:
# Scan all complete patient cases
patient_records = scan_brats_dataset(config.data_dir)
print(f"Total valid patient cases found: {len(patient_records)}")

if len(patient_records) == 0:
    raise FileNotFoundError(
        f"No patient cases found in '{config.data_dir}'. "
        "Please verify your dataset directory or set CUSTOM_DATASET_PATH in section 3."
    )

# Inspect a sample record
sample_patient = patient_records[0]
print(f"\nSample Patient ID: {sample_patient['patient_id']}")

# Volume statistics for FLAIR modality
flair_stats = compute_volume_statistics(sample_patient['flair'])
print("\nFLAIR Modality Metadata:")
for k, v in flair_stats.items():
    print(f"  {k}: {v}")

# Label distribution breakdown
label_stats = analyze_label_distribution(sample_patient['seg'])
print("\nSegmentation Label Breakdown:")
print(f"  Total Voxels: {label_stats['total_voxels']:,}")
print(f"  Background Voxels: {label_stats['background_voxels']:,} ({100*(1 - label_stats['wt_ratio']):.2f}%)")
print(f"  Necrotic Core (NCR - Label 1): {label_stats['ncr_voxels']:,}")
print(f"  Peritumoral Edema (ED - Label 2): {label_stats['ed_voxels']:,}")
print(f"  Enhancing Tumor (ET - Label 4): {label_stats['et_voxels']:,}")
print(f"  Whole Tumor (WT: 1+2+4): {label_stats['wt_voxels']:,} ({100*label_stats['wt_ratio']:.2f}%)")
print(f"  Tumor Core (TC: 1+4): {label_stats['tc_voxels']:,} ({100*label_stats['tc_ratio']:.2f}%)")

In [ ]:
# Visualize 4 MRI modalities and BraTS sub-region targets
eda_fig_path = os.path.join(config.output_dir, "eda_sample_patient.png")
fig = plot_patient_modalities(
    patient_record=sample_patient,
    slice_idx=None,  # Automatically selects slice with maximum tumor presence
    save_path=eda_fig_path
)
plt.show()

## 5. Dataset and DataLoaders Setup

Splits the dataset into training and validation sets.
- Training uses random 3D crops of size `(96, 96, 96)` and axis flips.
- Validation keeps full 3D volumes for sliding window inference.

In [ ]:
# Adjust batch size and workers adaptively based on hardware
batch_size = config.batch_size if torch.cuda.is_available() else 1
num_workers = config.num_workers if torch.cuda.is_available() else 0

# Optional limit on sample count for rapid experimentation (set None for full dataset)
MAX_SAMPLES = None

train_loader, val_loader, train_files, val_files = build_dataloaders(
    data_dir=config.data_dir,
    train_ratio=config.train_ratio,
    batch_size=batch_size,
    val_batch_size=config.val_batch_size,
    num_workers=num_workers,
    roi_size=config.roi_size,
    cache_rate=config.cache_rate,
    seed=config.seed,
    max_samples=MAX_SAMPLES
)

print(f"Training Patients: {len(train_files)} (Batches per epoch: {len(train_loader)})")
print(f"Validation Patients: {len(val_files)} (Batches per epoch: {len(val_loader)})")

In [ ]:
# Inspect a sample batch from training loader
sample_batch = next(iter(train_loader))
images = sample_batch["image"]
labels = sample_batch["label"]

print(f"Batch Image Shape: {images.shape}")   # Expected: (B, 4, 96, 96, 96)
print(f"Batch Label Shape: {labels.shape}")   # Expected: (B, 3, 96, 96, 96)
print(f"Image Intensity: min={images.min():.2f}, max={images.max():.2f}")
print(f"Label Dtype: {labels.dtype}, Target Values: {torch.unique(labels).tolist()}")

## 6. Model, Loss, Optimizer, and Metrics Setup

Instantiate the MONAI 3D UNet model, DiceCELoss, AdamW optimizer, and Cosine Annealing learning rate scheduler.

In [ ]:
# Instantiate 3D UNet
model = build_unet_model(
    spatial_dims=config.spatial_dims,
    in_channels=config.in_channels,
    out_channels=config.out_channels,
    channels=config.channels,
    strides=config.strides,
    num_res_units=config.num_res_units,
    norm=config.norm,
    dropout=config.dropout
).to(config.device)

# Model summary
summary = get_model_summary(model)
print(f"Architecture: {summary['model_class']}")
print(f"Total Parameters: {summary['total_parameters']:,}")
print(f"Model Size: {summary['model_size_mb']} MB")

# Loss function and evaluation metric
loss_fn = build_loss_fn(config.loss_type)
metric = BraTSMetric()

# Optimizer and Scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.lr,
    weight_decay=config.weight_decay
)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=config.max_epochs,
    eta_min=1e-6
)

print("Model, loss function, optimizer, and metrics successfully initialized.")

## 7. Pipeline Sanity Check

Execute one forward and backward pass with one sample batch to verify memory and gradient computation.

In [ ]:
model.train()
test_images = images[:1].to(config.device)
test_labels = labels[:1].to(config.device)

optimizer.zero_grad()
test_outputs = model(test_images)
test_loss = loss_fn(test_outputs, test_labels)
test_loss.backward()
optimizer.step()
optimizer.zero_grad()

print(f"Sanity check passed. Dummy batch loss: {test_loss.item():.4f}")
print(f"Output logits shape: {test_outputs.shape}")

## 8. Training and Validation Execution

Execute full training loop. Best model checkpoint is automatically saved based on Mean Dice.

In [ ]:
# Configure Trainer
trainer = BraTSTrainer(
    model=model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    config=config,
    lr_scheduler=lr_scheduler,
    metric=metric
)

# Set epochs to train (e.g. 50-100 on GPU)
EPOCHS_TO_TRAIN = config.max_epochs
VAL_INTERVAL = config.val_interval

# Run training
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    max_epochs=EPOCHS_TO_TRAIN,
    val_interval=VAL_INTERVAL
)

## 9. Training Curves

Plot learning progress across epochs: training loss and validation Dice scores (TC, WT, ET, Mean Dice).

In [ ]:
curves_save_path = os.path.join(config.output_dir, "training_curves.png")
fig_curves = plot_training_curves(
    history=trainer.history,
    save_path=curves_save_path
)
plt.show()

## 10. Qualitative Evaluation on Validation Case

Load the best model checkpoint and evaluate on a full 3D volume using sliding window inference.

In [ ]:
from monai.inferers import sliding_window_inference

# Load best checkpoint if available
best_ckpt_path = os.path.join(config.checkpoint_dir, "best_metric_model.pth")
if os.path.exists(best_ckpt_path):
    load_checkpoint(best_ckpt_path, model, device=config.device)
    print(f"Loaded best checkpoint from: {best_ckpt_path}")
else:
    print(f"Checkpoint not found at {best_ckpt_path}; evaluating current model weights.")

# Run sliding window inference on 1 validation case
model.eval()
val_batch = next(iter(val_loader))
val_image = val_batch["image"].to(config.device)
val_label = val_batch["label"].to(config.device)

with torch.no_grad():
    val_pred = sliding_window_inference(
        inputs=val_image,
        roi_size=config.roi_size,
        sw_batch_size=config.sw_batch_size,
        predictor=model,
        overlap=config.overlap,
        mode="gaussian"
    )
    val_pred_mask = (torch.sigmoid(val_pred) > 0.5).float()

# Convert to numpy for visualization
img_np = val_image[0].cpu().numpy()
gt_np = val_label[0].cpu().numpy()
pred_np = val_pred_mask[0].cpu().numpy()

# Generate qualitative comparison
val_vis_path = os.path.join(config.output_dir, "validation_prediction_sample.png")
fig_eval = visualize_predictions(
    image=img_np,
    label=gt_np,
    pred=pred_np,
    slice_idx=None,
    save_path=val_vis_path
)
plt.show()